# クロスエンコーダで検索結果を並べ替える (リランキング)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_search/rerank.ipynb)

NLP クックブック 20-12 ／ 検索

検索の上位候補を、より精度の高いモデルで並べ替えて、本当に関連する文書を一番上に持ってきたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("hotchpotch/japanese-reranker-cross-encoder-xsmall-v1", max_length=512)

query = "日本で一番高い山は何ですか？"
candidates = [
    "日本の山の標高ランキングは登山者の間でよく話題になる。",
    "エベレストは世界で最も高い山で、標高は 8849 メートルである。",
    "富士山は標高 3776 メートルで、日本の最高峰である。",
    "北岳は標高 3193 メートルで、富士山に次いで日本で 2 番目に高い。",
    "日本で一番長い川は信濃川である。",
]
scores = reranker.predict([(query, c) for c in candidates])
for s, c in sorted(zip(scores, candidates), reverse=True):
    print(f"{s:.3f}  {c}")

0.801  富士山は標高 3776 メートルで、日本の最高峰である。
0.697  北岳は標高 3193 メートルで、富士山に次いで日本で 2 番目に高い。
0.651  エベレストは世界で最も高い山で、標高は 8849 メートルである。
0.612  日本の山の標高ランキングは登山者の間でよく話題になる。
0.562  日本で一番長い川は信濃川である。


In [3]:
# rank() を使うと並べ替えまでやってくれる
for r in reranker.rank(query, candidates, top_k=2, return_documents=True):
    print(r["corpus_id"], round(float(r["score"]), 3), r["text"])

2 0.801 富士山は標高 3776 メートルで、日本の最高峰である。
3 0.697 北岳は標高 3193 メートルで、富士山に次いで日本で 2 番目に高い。


- バイエンコーダ (文埋め込み) はクエリと文書を別々にベクトル化するので速いが粗い。クロスエンコーダは両方を連結して入力するので正確だが遅い。
- 定番の構成: BM25 や埋め込み検索で上位 100 件を取得 → クロスエンコーダで上位 10 件に絞る。
- 多言語なら `BAAI/bge-reranker-v2-m3`、日本語で高精度なら `hotchpotch/japanese-reranker-cross-encoder-large-v1` など。